# Interactive region annotation

This tutorial shows how to use `sdata.pl.annotate()` to draw regions of interest directly on a `spatialdata-plot` canvas inside a notebook and persist them as a `ShapesModel` element. The widget is a custom [anywidget] that draws client-side, so it works over SSH (Jupyter or VSCode-Remote-SSH) without streaming PNG frames per mouse-move.

**Dataset**: a Visium H&E mouse brain section, downloaded by `squidpy.datasets.visium_hne_sdata` from the scverse example data host. The download (~400 MB) is cached after the first run.

**Requires the `interactive` extra:**

```bash
pip install 'spatialdata-plot[interactive]'
```

[anywidget]: https://anywidget.dev

## Loading the dataset

`squidpy.datasets.visium_hne_sdata()` returns a `SpatialData` object with the multi-resolution H&E image (`'hne'`) and the spot polygons (`'spots'`), both aligned in the `'global'` coordinate system.

In [ ]:
import squidpy as sq
from shapely.geometry import Polygon

import spatialdata as sd
import spatialdata_plot  # noqa: F401  (registers the .pl accessor)
from spatialdata.models import ShapesModel
from spatialdata.transformations.transformations import Identity

sdata = sq.datasets.visium_hne_sdata()
sdata

## Inspect what we'll annotate

Before drawing, let's render the image so we know what to outline.

In [ ]:
sdata.pl.render_images("hne").pl.show()

## Launching the widget

Call `sdata.pl.annotate(coordinate_system, element)` to open the drawing canvas. The image is rendered once via the standard `render_images` pipeline, exported to PNG, and laid under a client-side SVG drawing surface; all interaction happens in the browser, and only the final shape geometry round-trips to the kernel on Save.

```python
sdata.pl.annotate(
    coordinate_system="global",
    element="hne",
    max_width=880,    # display hint; underlying render is 840×840
    persist=True,     # shows a "Write to disk" button
)
```

> The code above is intentionally a Markdown block, not a code cell — the widget needs a live JS runtime, which the static docs build can't provide. Run the line in your own notebook (Jupyter Lab or VSCode-Remote-SSH) to see the canvas.

**Drawing tools**: rectangle (drag), polygon (click vertices, snap-to-first or Enter to close), lasso (drag freehand).  
**Shortcuts**: `R` / `P` / `L` switch tool · wheel zoom · Shift+drag pan · Alt+click a shape to delete · Ctrl+Z undo · `F` fit · `Esc` cancel in-progress shape.

When you click **Save** the shapes on the canvas are committed to `sdata.shapes[<name>]` as a single `ShapesModel` (multiple rows if you drew multiple shapes). The optional **Write to disk** button calls `sdata.write_element(<name>)` to persist to the backing zarr.

![Drawing a region with sdata.pl.annotate](interactive_annotate.gif)

*Live recording of the widget. Replace this file with your own capture before publishing the tutorial.*

## What the widget produces

To keep the rest of this notebook reproducible without a live kernel, the next cell creates the same `ShapesModel` the widget would have written if you'd drawn a polygon around the hippocampus and clicked Save with the name `'tumor_region'`. After this cell, downstream code is identical regardless of whether you ran the widget or this simulated commit.

In [ ]:
# Pretend the user drew this polygon in the widget and clicked Save with
# name="tumor_region". Coordinates are in the 'global' coordinate system
# of the visium_hne_sdata dataset.
hippocampus_polygon = Polygon(
    [
        (3200, 4800),
        (4800, 4400),
        (5600, 5200),
        (5400, 6400),
        (4200, 6600),
        (3200, 6000),
    ]
)

import geopandas as gpd

sdata.shapes["tumor_region"] = ShapesModel.parse(
    gpd.GeoDataFrame({"geometry": [hippocampus_polygon]}),
    transformations={"global": Identity()},
)
sdata.shapes["tumor_region"]

## Working with the saved region

The committed element is a normal `ShapesModel` — every downstream API in `spatialdata` and `spatialdata-plot` treats it like any other shapes layer. Here we overlay it on the H&E to confirm placement.

In [ ]:
(
    sdata
    .pl.render_images("hne")
    .pl.render_shapes("tumor_region", outline_color="#22d3ee", fill_alpha=0.2)
    .pl.show()
)

## Cropping the dataset to the region

Because the region is a registered `ShapesModel`, `sdata.query.polygon` can subset every element down to what falls inside it. Useful for focused analyses or quick QC on a single anatomical structure.

In [ ]:
subset = sd.polygon_query(
    sdata,
    sdata["tumor_region"],
    target_coordinate_system="global",
)
subset

In [ ]:
(
    subset
    .pl.render_images("hne")
    .pl.render_shapes("spots", fill_alpha=0.4)
    .pl.show()
)

## For reproducibility

In [ ]:
# ruff: noqa: F401, F811, I001, E402
# fmt: off
import spatialdata_plot

%load_ext watermark
# fmt: on

%watermark -v -m -p spatialdata,spatialdata_plot,squidpy,anywidget,matplotlib,numpy